# Exploración HSV masked RoCoLe

Este EDA mira solo la tabla HSV masked. Objetivo: ver si los dieciséis bins HSV enmascarados separan sanas de afectadas en train (calidad, univariados de pocos bins, clases y correlaciones) y decidir con honestidad si vale la pena un 04 con RGB/GLCM más HSV, solo HSV, o aparcar la familia.


## Rutas y semilla

Apuntamos al CSV hsv_hist_features_masked y a la carpeta de figuras. Semilla 42 por si hay muestreos visuales.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path


In [ ]:
# Semilla fija para muestreos visuales reproducibles
seed = 42
repo_root = Path("../..").resolve()
features_path = repo_root / "data" / "featured" / "rocole" / "hsv_hist_features_masked.csv"
results_dir = repo_root / "experiments" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

print("seed:", seed)
print("features_path:", features_path)
print("existe:", features_path.exists())
print("results_dir:", results_dir)
seed, features_path, results_dir


## Cargar tabla HSV

Cargamos la mesa HSV: identidad, dieciséis bins y error. Es la única fuente de este cuaderno.


In [ ]:
hsv_leaves = pd.read_csv(features_path)

hsv_leaves.head()


## Inspección de la tabla

Miramos cuántas filas y columnas hay, y las primeras filas, para confirmar que el inventario llegó entero.


In [ ]:
# hsv_leaves

print("forma:", hsv_leaves.shape)
print("columnas:", list(hsv_leaves.columns))

hsv_leaves.head()


## Calidad y errores

Contamos errores de máscara o lectura y cuántas filas tienen bins vacíos. La calidad condiciona cuánto podemos confiar en el solape.


In [ ]:
error_series = hsv_leaves["error"].fillna("").astype(str).str.strip()
extraction_error_count = int((error_series != "").sum())
hsv_bin_guess = [
    c
    for c in hsv_leaves.columns
    if c.startswith("hsv_h_b") or c.startswith("hsv_s_b") or c.startswith("hsv_v_b")
]
null_counts_bins = (
    hsv_leaves[hsv_bin_guess].isna().sum() if hsv_bin_guess else pd.Series(dtype=int)
)
rows_with_null_bins = (
    int(hsv_leaves[hsv_bin_guess].isna().any(axis=1).sum()) if hsv_bin_guess else 0
)
quality_summary = {
    "n_rows": int(len(hsv_leaves)),
    "extraction_error_count": extraction_error_count,
    "rows_with_null_bins": rows_with_null_bins,
    "null_counts_bins": null_counts_bins.to_dict() if len(null_counts_bins) else {},
}

print("filas:", quality_summary["n_rows"])
print("filas con error de extracción:", quality_summary["extraction_error_count"])
print("filas con algún bin nulo:", quality_summary["rows_with_null_bins"])
print("nulos por bin:", quality_summary["null_counts_bins"])
quality_summary


Separamos las dieciséis columnas de histograma del resto. Son las medidas; la etiqueta sigue en la misma fila. No codificamos la etiqueta todavía.


In [ ]:
# hsv_leaves

hsv_feature_columns = [
    "hsv_h_b0",
    "hsv_h_b1",
    "hsv_h_b2",
    "hsv_h_b3",
    "hsv_h_b4",
    "hsv_h_b5",
    "hsv_h_b6",
    "hsv_h_b7",
    "hsv_s_b0",
    "hsv_s_b1",
    "hsv_s_b2",
    "hsv_s_b3",
    "hsv_v_b0",
    "hsv_v_b1",
    "hsv_v_b2",
    "hsv_v_b3",
]
missing_feature_columns = [c for c in hsv_feature_columns if c not in hsv_leaves.columns]

print("columnas HSV pedidas:", len(hsv_feature_columns))
print("faltan en la tabla:", missing_feature_columns)
hsv_feature_columns


Apartamos solo las hojas de aprender. Si hay error de extracción, las filtramos aquí. Clases, solapes y correlaciones se miran en train; el test no decide.


In [ ]:
# hsv_leaves

error_mask = hsv_leaves["error"].fillna("").astype(str).str.strip() != ""
train_leaves = hsv_leaves.loc[
    (hsv_leaves["split"] == "train") & (~error_mask)
].copy()

print("train filas (sin error):", len(train_leaves))
print("excluidas por error en todo el CSV:", int(error_mask.sum()))
train_leaves.head()


Elegimos pocos cajones representativos —dos de matiz, uno de saturación y uno de valor— para no ahogarnos en dieciséis gráficos. Todavía no dibujamos ni correlacionamos todo.


In [ ]:
# hsv_feature_columns

# Dos de H, uno de S y uno de V: cobertura mínima sin saturar figuras
representative_bins = ["hsv_h_b0", "hsv_h_b3", "hsv_s_b1", "hsv_v_b2"]

print("representative_bins:", representative_bins)
representative_bins


Miramos la distribución de esos pocos bins en train, sin cortar outliers ni elegir umbral. Figuras con prefijo rocole-hsv-masked-.


In [ ]:
fig, axes = plt.subplots(1, len(representative_bins), figsize=(14, 4))
for axis, column in zip(axes, representative_bins):
    sns.boxplot(y=train_leaves[column], ax=axis, color="steelblue")
    axis.set_title(column)
fig.suptitle("Cajas univariadas HSV (train, máscara COCO)")
fig.tight_layout()
univariate_box_path = results_dir / "rocole-hsv-masked-representative-boxplots.png"
fig.savefig(univariate_box_path, dpi=120, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, len(representative_bins), figsize=(14, 4))
row_index = np.arange(len(train_leaves))
for axis, column in zip(axes, representative_bins):
    axis.scatter(row_index, train_leaves[column], s=8, alpha=0.5)
    axis.set_xlabel("índice de fila")
    axis.set_ylabel(column)
    axis.set_title(column)
fig.suptitle("Dispersión univariada HSV (train, máscara COCO)")
fig.tight_layout()
univariate_scatter_path = results_dir / "rocole-hsv-masked-representative-scatter.png"
fig.savefig(univariate_scatter_path, dpi=120, bbox_inches="tight")
plt.show()

univariate_figures = [univariate_box_path, univariate_scatter_path]

print("figuras guardadas:")
for path in univariate_figures:
    print(" ", path)
univariate_figures


Comparamos esos mismos bins entre sanas y afectadas, solo en train, para ver si hay separación a ojo. No entrenamos ni miramos el test.


In [ ]:
# train_leaves, representative_bins, results_dir

fig, axes = plt.subplots(1, len(representative_bins), figsize=(14, 4))
for axis, column in zip(axes, representative_bins):
    sns.boxplot(data=train_leaves, x="Binary.Label", y=column, ax=axis)
    axis.set_title(column)
fig.suptitle("Bins HSV por Binary.Label (train, máscara COCO)")
fig.tight_layout()
class_compare_path = results_dir / "rocole-hsv-masked-representative-by-label-boxplots.png"
fig.savefig(class_compare_path, dpi=120, bbox_inches="tight")
plt.show()

class_compare_figures = [class_compare_path]

print("figura guardada:", class_compare_path)
class_compare_figures


Nos fijamos en el solape entre sanas y afectadas: si las cajas se montan casi del todo, la señal es débil aunque haya diferencia de medias. No pasamos a modelo ni usamos Multiclass.Label como color.


In [ ]:
# train_leaves, representative_bins

overlap_rows = []
for column in representative_bins:
    for label_name, group in train_leaves.groupby("Binary.Label"):
        values = group[column].dropna()
        overlap_rows.append(
            {
                "bin": column,
                "Binary.Label": label_name,
                "n": int(len(values)),
                "q25": float(values.quantile(0.25)) if len(values) else np.nan,
                "median": float(values.median()) if len(values) else np.nan,
                "q75": float(values.quantile(0.75)) if len(values) else np.nan,
                "min": float(values.min()) if len(values) else np.nan,
                "max": float(values.max()) if len(values) else np.nan,
            }
        )
overlap_notes = pd.DataFrame(overlap_rows)

# Tras correr: anotar a ojo si el IQR sana/afectada se monta casi entero
print(overlap_notes.to_string(index=False))
overlap_notes


Pasamos la etiqueta binaria a números (sana 0, afectada 1) solo para correlacionar, sin entrenar. Sin severidad ni el split de test.


In [ ]:
# train_leaves

binary_label_code = train_leaves["Binary.Label"].map(
    {"healthy": 0, "unhealthy": 1}
)

print(binary_label_code.value_counts(dropna=False))
binary_label_code


Correlación de cada bin HSV con el código de etiqueta en train. Miramos magnitud y signo; no es ranking de predictoras definitivo. No ajustamos modelo ni elegimos columnas con acierto en test.


In [ ]:
# train_leaves, hsv_feature_columns, binary_label_code

feature_label_correlation = train_leaves[hsv_feature_columns].corrwith(
    binary_label_code
).sort_values(key=lambda s: s.abs(), ascending=False)

print(feature_label_correlation)
feature_label_correlation


Con lo visto (calidad, solape, correlaciones) decidimos con honestidad: ¿sumar HSV a las siete RGB/GLCM masked, probar solo HSV, o aparcar? Un 04-hsv solo si hay señal clara. No entrenamos baseline en este cuaderno.


In [ ]:
# quality_summary, overlap_notes, feature_label_correlation

# Evidencia del run (train, 1037 hojas, 0 errores):
# - mejor |corr| con etiqueta: hsv_h_b0 ≈ 0.26; hsv_v_b3 ≈ -0.15; hsv_h_b7 ≈ 0.15
# - solape sano/afectado sigue grande en cajas (IQR se montan)
# - señal moderada, comparable a homogeneidad GLCM masked; no basta sola
hsv_go_decision = "sumar_hsv_a_rgb_glcm"

print("hsv_go_decision:", hsv_go_decision)
print(
    "Motivo: matiz aporta señal moderada y probablemente complementaria "
    "a las 7 RGB/GLCM masked; aparcar sería prematuro; solo-HSV no tiene soporte."
)
hsv_go_decision


## Conclusiones

Inventario HSV masked: 1481 filas, 0 errores, 16 bins que suman 1 por canal.

En train, la mejor correlación con la etiqueta es hsv_h_b0 (cerca de 0.26). Hay señal, pero el solape sana/afectada sigue grande: no es un separador limpio por sí solo.

Decisión: sumar_hsv_a_rgb_glcm. Tiene sentido probar un baseline que una las 7 medidas RGB/GLCM masked con estos 16 bins, eligiendo en val. No abrir un baseline solo-HSV todavía.

Los cuadernos 01–04 publicados (full y masked RGB/GLCM) no se tocan; esto sigue siendo rama de desarrollo.
